# OVD-Diagnose — Domain 2: Medical (VinDr chest X-ray)

Second specialized domain for the cross-domain fingerprint. VinDr = 14 finding classes with
boxes → exercises the semantic-confusion axis (unlike single-class fish/fruit detection).
Expected contrast vs aerial: subtle low-contrast findings + weak CLIP semantics for medical
terms → likely **L / C-bound** rather than aerial's S-bound.

**Data:** Kaggle → Add Data → `awsaf49/vinbigdata-512-image-dataset` (512px PNGs + train.csv).

In [ ]:
import torch
print('torch', torch.__version__, '| cuda', torch.cuda.is_available(), '| GPUs', torch.cuda.device_count())

In [ ]:
REPO = 'https://github.com/ShMazumder/YOLOBERT.git'
import os
if not os.path.isdir('/kaggle/working/YOLOBERT'):
    !git clone $REPO /kaggle/working/YOLOBERT
%cd /kaggle/working/YOLOBERT
!git pull -q || true
!pip install -q ultralytics transformers pycocotools pyyaml pillow

In [ ]:
# Paths for awsaf49/vinbigdata-512-image-dataset (adjust if your mount differs).
CSV  = '/kaggle/input/datasets/awsaf49/vinbigdata-512-image-dataset/vinbigdata/train.csv'
IMGS = '/kaggle/input/datasets/awsaf49/vinbigdata-512-image-dataset/vinbigdata/train'
OUT  = 'data/medical/annotations/instances_val.json'
import os
print('csv exists:', os.path.exists(CSV), '| imgs exists:', os.path.isdir(IMGS))
print('csv header:', open(CSV).readline().strip())

In [ ]:
# Convert VinDr train.csv -> COCO json (14 finding classes)
!python tools/vindr_to_coco.py --csv "{CSV}" --imgs "{IMGS}" --out "{OUT}" --img_ext .png

In [ ]:
# BOX-ALIGNMENT CHECK — the one medical risk (512-resize vs original DICOM coords).
from pycocotools.coco import COCO
c = COCO(OUT)
bb = [a['bbox'] for a in c.loadAnns(c.getAnnIds())[:8]]
print('sample boxes:', [[round(v) for v in b] for b in bb])
print('img sizes   :', [(i['width'], i['height']) for i in c.loadImgs(c.getImgIds()[:3])])
mx = max(v for b in bb for v in b)
print('max box coord:', round(mx),
      '-> OK (scaled)' if mx <= 600 else '-> UNSCALED: boxes in original DICOM px, need --scale_from_dims')

If the check says **OK (scaled)**, run the models. If **UNSCALED**, stop — we need a dims source
(the dataset's meta csv with original width/height) to rescale; paste the CSV header and I'll patch the converter.

In [ ]:
# Run all models on the medical domain (same driver as aerial)
LIMIT = 200   # 0 = full; 200 for a first pass
limit_arg = f'--limit {LIMIT}' if LIMIT else ''
!python tools/run_all.py \
  --ann  data/medical/annotations/instances_val.json \
  --imgs "{IMGS}" \
  --out  runs/diag/medical \
  --device cuda:0 {limit_arg} \
  --models "yoloworld:yolov8s-world.pt,owlv2:google/owlv2-base-patch16-ensemble,groundingdino:IDEA-Research/grounding-dino-tiny" \
  --sam_weights mobile_sam.pt

In [ ]:
import pandas as pd
df = pd.read_csv('runs/diag/medical/fingerprints.csv')
print(df.to_string(index=False))

**Cross-domain read:** compare to `runs/diag/aerial/fingerprints.csv`. Aerial S-bound + medical
L/C-bound = the paper's core divergence figure. Medical also S-bound = a cross-domain vocab trend
(still a finding). Either way, two domains × 3 models = the fingerprint table for the paper.